<figure>
<center>
<img src="https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png"/>
</center></figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**
### **Métodos Predictivos**
### Cátedra: Bianco
#### **Conjuntos de datos desbalanceados**

In [1]:
import pandas as pd
import sklearn as sk
from sklearn import model_selection
from sklearn import metrics
from sklearn import tree

import warnings
warnings.filterwarnings("ignore")

from collections import Counter

import imblearn as im
from imblearn import under_sampling
from imblearn import over_sampling

In [3]:
import os
os.chdir(os.path.dirname(__vsc_ipynb_file__))

# datos: https://www.kaggle.com/datasets/mathchi/diabetes-data-set
df = pd.read_csv('diabetes.csv')
df = df.sample(frac=1).reset_index(drop=True) # mezcla los datos
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0,167,0,0,0,32.3,0.839,30,1
1,2,105,58,40,94,34.9,0.225,25,0
2,1,135,54,0,0,26.7,0.687,62,0
3,7,100,0,0,0,30.0,0.484,32,1
4,13,106,70,0,0,34.2,0.251,52,0


In [4]:
df['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [5]:
def accuracy_precision_recall(X, y):
    X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.5, shuffle=True, stratify=y, random_state=42)

    clf = sk.tree.DecisionTreeClassifier(random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print("accuracy:", sk.metrics.accuracy_score(y_test, y_pred))
    print("precision:", sk.metrics.precision_score(y_test, y_pred))
    print("recall:", sk.metrics.recall_score(y_test, y_pred))
    print("f1_score:", sk.metrics.f1_score(y_test, y_pred))

X = df[df.columns.drop("Outcome")]
y = df["Outcome"]
accuracy_precision_recall(X, y)

accuracy: 0.7005208333333334
precision: 0.5855855855855856
recall: 0.48507462686567165
f1_score: 0.5306122448979592


# Peso de la clase minoritaria

In [6]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.5, shuffle=True, stratify=y, random_state=42)
clf = sk.tree.DecisionTreeClassifier(class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("accuracy:", sk.metrics.accuracy_score(y_test, y_pred))
print("precision:", sk.metrics.precision_score(y_test, y_pred))
print("recall:", sk.metrics.recall_score(y_test, y_pred))
print("f1_score:", sk.metrics.f1_score(y_test, y_pred))

accuracy: 0.6796875
precision: 0.5447154471544715
recall: 0.5
f1_score: 0.5214007782101168


# Métodos de submuestreo

## Submuestreo al azar

Este procedimiento constituye el enfoque más directo para resolver el problema del desbalanceo de clases, el cual se presenta cuando la clase de interés (minoritaria) se encuentra fuertemente superada en cantidad por la otra categoría (mayoritaria).

A diferencia de los métodos basados en agrupamiento geométrico, esta técnica realiza una selección probabilística directa para equiparar los volúmenes de las muestras.

**Proceso Algorítmico Paso a Paso:**

1. **Separación de la Base:** Se dividen las características predictoras ($X$) de la variable objetivo que se desea predecir ($y$, denominada aquí `"Outcome"`).
2. **Identificación del Umbral ($N$):** El algoritmo contabiliza la cantidad total de registros disponibles en la clase minoritaria. Este número establecerá el tamaño objetivo ($N$) para la clase mayoritaria.
3. **Eliminación Aleatoria Simple:** Se seleccionan al azar observaciones de la clase mayoritaria y se eliminan del conjunto de datos, repitiendo el proceso hasta que dicha categoría se reduce exactamente a un tamaño de $N$ registros. Esto se ejecuta de forma automática mediante la instrucción `sampling_strategy='majority'`.
4. **Fijación de la Semilla:** El parámetro `random_state=42` asegura la reproducibilidad del experimento.

---

**Lectura del Resultado Final**

La instrucción `print(Counter(y))` muestra en la consola la distribución final de las clases:

$$\text{Counter}(\{0: 100, 1: 100\})$$

> **Ventaja y Desventaja Metodológica:** La principal ventaja de este método es su bajísimo costo computacional y su velocidad de ejecución. La desventaja crítica es el riesgo de **pérdida de información**, ya que se eliminan datos reales de forma aleatoria que podrían contener patrones valiosos.

In [7]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

cc = im.under_sampling.RandomUnderSampler(sampling_strategy='majority', random_state=42)

X, y = cc.fit_resample(X, y)

print(Counter(y))

Counter({0: 268, 1: 268})


In [8]:
accuracy_precision_recall(X, y)

accuracy: 0.6716417910447762
precision: 0.6916666666666667
recall: 0.6194029850746269
f1_score: 0.6535433070866141


## Submuestro por centroides de clusters

Este procedimiento se utiliza para resolver el problema del desbalanceo de clases, situación que ocurre cuando una categoría cuenta con una cantidad de registros significativamente mayor (clase mayoritaria) en comparación con la otra (clase minoritaria).

En lugar de eliminar observaciones de la clase mayoritaria de forma aleatoria —lo que provocaría una pérdida de información valiosa—, se aplica una técnica de **submuestreo inteligente (Under-sampling)** basada en algoritmos de agrupamiento (*clustering*).

**Proceso Algorítmico Paso a Paso:**

1. **Separación de la Base:** Se dividen las características predictoras ($X$) de la variable objetivo ($y$).
2. **Agrupamiento Geométrico (Clustering):** El algoritmo analiza el espacio multidimensional y agrupa los datos de la clase mayoritaria en $k$ conglomerados usando $K$-Means.
3. **Sustitución por el Centroide:** Se calcula el centro geométrico de cada grupo. Se eliminan todos los puntos reales del conglomerado y se conservan únicamente las coordenadas del punto central (centroide sintético).
4. **Equiparación de las Clases:** El proceso se repite hasta que la cantidad de centroides de la clase mayoritaria iguala exactamente a la cantidad de datos originales de la clase minoritaria.

---

**Lectura del Resultado Final**

$$\text{Counter}(\{0: 100, 1: 100\})$$

> **Ventaja Metodológica:** Al utilizar los centros de los grupos, no se destruye la estructura ni la densidad de la distribución original de los datos mayoritarios. Se realiza una reducción basada en criterios geométricos estrictos y no en el azar.

In [9]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

cc = im.under_sampling.ClusterCentroids(sampling_strategy='majority', random_state=42)

X, y = cc.fit_resample(X, y)

print(Counter(y))

Counter({0: 268, 1: 268})


In [10]:
accuracy_precision_recall(X, y)

accuracy: 0.6492537313432836
precision: 0.6470588235294118
recall: 0.6567164179104478
f1_score: 0.6518518518518519


## Submuestreo NearMiss

El algoritmo `NearMiss` constituye una técnica de submuestreo inteligente que utiliza la lógica del algoritmo de los $k$ vecinos más cercanos ($k$-NN) para reducir el volumen de la clase mayoritaria. El parámetro `version` permite parametrizar tres heurísticas distintas:

---

**NearMiss Versión 1: Mínima Distancia Promedio (Frontera Cercana)**

Conserva aquellas observaciones de la clase mayoritaria cuya **distancia promedio a los $k$ vecinos más cercanos de la clase minoritaria sea la más corta**.

**NearMiss Versión 2: Máxima Distancia Promedio (Frontera Amplia)**

Conserva aquellas observaciones de la clase mayoritaria cuya **distancia promedio a los $k$ vecinos más lejanos de la clase minoritaria sea la más corta**.

**NearMiss Versión 3: Selección por Vecindad Directa**

Funciona en dos etapas: primero preselecciona los vecinos más cercanos de la clase mayoritaria para cada observación minoritaria; luego conserva únicamente los que tienen la **menor distancia promedio a sus propios vecinos de la clase minoritaria**.

---

> **Consideración Metodológica Crítica:** Las versiones 1 y 3 son altamente vulnerables a la presencia de ruido o valores atípicos (*outliers*). La versión 2 tiende a generar fronteras de decisión más suaves y estables.

In [11]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

nm1 = im.under_sampling.NearMiss(sampling_strategy='majority', version=1)
X,  y = nm1.fit_resample(X, y)
print(Counter(y))

Counter({0: 268, 1: 268})


In [12]:
accuracy_precision_recall(X, y)

accuracy: 0.6753731343283582
precision: 0.688
recall: 0.6417910447761194
f1_score: 0.6640926640926641


In [13]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

nm2 = im.under_sampling.NearMiss(sampling_strategy='majority', version=2)
X,  y = nm2.fit_resample(X, y)
print(Counter(y))

Counter({0: 268, 1: 268})


In [14]:
accuracy_precision_recall(X, y)

accuracy: 0.7276119402985075
precision: 0.7401574803149606
recall: 0.7014925373134329
f1_score: 0.7203065134099617


In [15]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

nm3 = im.under_sampling.NearMiss(sampling_strategy='majority', version=3)
X,  y = nm3.fit_resample(X, y)
print(Counter(y))

Counter({0: 268, 1: 268})


In [16]:
accuracy_precision_recall(X, y)

accuracy: 0.5932835820895522
precision: 0.5862068965517241
recall: 0.6343283582089553
f1_score: 0.6093189964157706


## Enlaces Tomek

Este procedimiento aplica un enfoque de submuestreo selectivo y guiado por la proximidad geométrica de los datos. Los **Enlaces de Tomek** se centran en **limpiar la zona fronteriza de decisión** para eliminar el ruido y definir con mayor claridad los límites entre ambas categorías.

Dados dos ejemplos, $x_i$ (clase mayoritaria) y $x_j$ (clase minoritaria), existe un **Enlace de Tomek** entre ellos si se cumple la condición de vecindad mutua más cercana: son sus respectivos vecinos más cercanos entre sí.

---

**Proceso Algorítmico Paso a Paso:**

1. **Separación de la Base:** Se dividen las características predictoras ($X$) de la variable objetivo ($y$).
2. **Identificación de Parejas Críticas:** El algoritmo detecta todos los pares de puntos opuestos que califican como Enlaces de Tomek.
3. **Eliminación Selectiva:** Con `sampling_strategy='majority'`, se elimina únicamente la observación de la clase mayoritaria de cada par detectado.
4. **Suavizado de la Frontera:** Se incrementa el espacio de separación entre ambas clases.

---

**Lectura del Resultado Final**

Este algoritmo **casi nunca dejará un balance simétrico perfecto** ($50\% - 50\%$):

$$\text{Counter}(\{0: 942, 1: 100\})$$

> **Consideración Metodológica Crítica:** Este método no busca resolver el desbalance numérico por completo, sino mejorar la calidad geométrica de los datos. Es muy común utilizarlo como paso de "limpieza final" combinado con otras técnicas de sobremuestreo.

In [17]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

tl = im.under_sampling.TomekLinks(sampling_strategy='majority')

X, y = tl.fit_resample(X, y)

print(Counter(y))

Counter({0: 445, 1: 268})


In [18]:
accuracy_precision_recall(X, y)

accuracy: 0.742296918767507
precision: 0.65
recall: 0.6791044776119403
f1_score: 0.6642335766423357


## Submuestro por vecinos cercanos editados

Este procedimiento implementa una estrategia de submuestreo inteligente basada en el filtrado de ruido. `EditedNearestNeighbours` (ENN) examina la consistencia local de los datos y elimina aquellas observaciones de la clase mayoritaria que contradicen a su entorno geométrico.

**Regla de Decisión Matemática:**
Para cada observación $x_i$ de la clase mayoritaria, el algoritmo identifica sus $k$ vecinos más cercanos (por defecto, $k = 3$). Se evalúa la clase dominante dentro de esa vecindad:

- Si la clase de $x_i$ **no coincide** con la clase mayoritaria de sus propios vecinos, el punto es clasificado como **inconsistente o ruido** y es eliminado.

---

**Proceso Algorítmico Paso a Paso:**

1. **Separación de la Base:** Se dividen $X$ e $y$.
2. **Construcción de Vecindades ($k$-NN):** Se establecen los 3 vecinos más cercanos para cada punto.
3. **Validación de Consistencia:** Se identifican los puntos de la clase mayoritaria "aislados" o inmersos en territorio de la clase minoritaria.
4. **Remoción de Ruido:** Se eliminan los elementos discordantes, generando una frontera de decisión más limpia.

---

**ENN es un método de limpieza selectivo, no busca un balance $50\% - 50\%$:**

$$\text{Counter}(\{0: 876, 1: 100\})$$

> **Consideración Metodológica Crítica:** Al eliminar los ejemplos de la clase mayoritaria que causan ambigüedad, ENN previene eficazmente el sobreajuste (*overfitting*) y permite que la clase minoritaria sea delimitada con mayor precisión.

In [19]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

enn = im.under_sampling.EditedNearestNeighbours(sampling_strategy='majority')

X, y = enn.fit_resample(X, y)

print(Counter(y))

Counter({1: 268, 0: 240})


In [20]:
accuracy_precision_recall(X, y)

accuracy: 0.7598425196850394
precision: 0.7703703703703704
recall: 0.7761194029850746
f1_score: 0.7732342007434945


## Submuestreo vecinos cercanos condensados

Este procedimiento implementa una estrategia basada en la eliminación de datos redundantes. `CondensedNearestNeighbour` (CNN) busca encontrar un **subconjunto mínimo de la clase mayoritaria** que sea capaz de clasificar correctamente al resto de los datos sin perder precisión.

**Proceso Algorítmico Paso a Paso:**

1. **Separación de la Base:** Se dividen $X$ e $y$.
2. **Inicialización de Almacenes:** Se crean el *Conjunto Almacén* ($U$) —con toda la clase minoritaria y un solo miembro aleatorio de la mayoritaria— y el *Conjunto de Prueba* ($V$) con el resto de la clase mayoritaria.
3. **Clasificación Iterativa (1-NN):** Para cada elemento de $V$: si se clasifica correctamente usando $U$, se descarta (redundante); si se clasifica incorrectamente, se transfiere a $U$ (informativo).
4. **Finalización:** El proceso cicla hasta que no se produzcan más transferencias hacia $U$.

---

**Este método suele reducir drásticamente la clase mayoritaria:**

$$\text{Counter}(\{0: 64, 1: 100\})$$

> **Consideración Metodológica Crítica:** La principal ventaja de CNN es la enorme reducción del tamaño del dataset. Sin embargo, es **altamente sensible al ruido**: un solo *outlier* de la clase mayoritaria en zona errónea será retenido obligatoriamente por el algoritmo.

In [21]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

cnn = im.under_sampling.CondensedNearestNeighbour(random_state=42)

X,  y = cnn.fit_resample(X, y)

print(Counter(y))

Counter({1: 268, 0: 188})


In [22]:
accuracy_precision_recall(X, y)

accuracy: 0.5921052631578947
precision: 0.6541353383458647
recall: 0.6492537313432836
f1_score: 0.651685393258427


# Métodos de sobremuestreo

## Sobremuestreo al azar

Este procedimiento implementa una estrategia de **sobremuestreo (Over-sampling)** que incrementa el tamaño de la clase minoritaria duplicando sus observaciones. La particularidad de esta configuración es el parámetro `shrinkage`, que transforma el duplicado exacto en una generación de datos sintéticos sutilmente modificados.

**Mecánica Matemática del Suavizado (Shrinkage):**

Con `shrinkage=0.1`, el algoritmo añade una pequeña cantidad de ruido gaussiano a las características continuas de las nuevas muestras:

$$X_{\text{nuevo}} = X_{\text{original}} + \epsilon \cdot \sigma \cdot s$$

Donde $\sigma$ es la desviación estándar de la característica, $s$ es el factor de suavizado (`shrinkage=0.1`) y $\epsilon \sim \mathcal{N}(0, 1)$.

---

**Resultado Final:** balance simétrico perfecto:

$$\text{Counter}(\{0: 1000, 1: 1000\})$$

> **Consideración Metodológica Crítica:** El parámetro `shrinkage` mitiga el riesgo de que los modelos memoricen los mismos puntos exactos, forzando fronteras de decisión más suaves y con mayor capacidad de generalización.

In [23]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

ros = im.over_sampling.RandomOverSampler(random_state=42, shrinkage=0.1)

X, y = ros.fit_resample(X, y)

print(Counter(y))

Counter({1: 500, 0: 500})


In [24]:
accuracy_precision_recall(X, y)

accuracy: 0.738
precision: 0.7279693486590039
recall: 0.76
f1_score: 0.7436399217221135


## Sobremuestreo SMOTE

Sobremuestreo Sintético por Vecinos Cercanos (`SMOTE`). A diferencia del muestreo aleatorio simple —que duplica filas existentes—, este algoritmo genera **observaciones completamente nuevas y sintéticas** basadas en la estructura matemática de la clase minoritaria.

**Mecánica Geométrica de la Generación de Datos:**

El algoritmo opera bajo una lógica de interpolación lineal entre puntos vecinos. Para cada observación de la clase minoritaria:

1. Se seleccionan sus $k$ vecinos más cercanos (por defecto, $k = 5$) de la misma clase.
2. Se elige uno de estos vecinos al azar.
3. Se genera un nuevo punto sintético en una posición intermedia del segmento que los une:

$$X_{\text{sintético}} = X_{\text{original}} + \lambda \cdot (X_{\text{vecino}} - X_{\text{original}})$$

Donde $\lambda \in [0, 1]$ es un número aleatorio uniforme.

---

**Resultado Final:** balance simétrico perfecto:

$$\text{Counter}(\{0: 1000, 1: 1000\})$$

> **Consideración Metodológica Crítica:** Al rellenar los espacios vacíos entre las observaciones existentes, SMOTE fuerza fronteras de decisión más amplias y continuas para la clase minoritaria, disminuyendo el sobreajuste. Se debe tener precaución si el dataset tiene mucho ruido, ya que los *outliers* minoritarios generarán "puentes" artificiales que desdibujan la frontera real.

In [25]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

smote = im.over_sampling.SMOTE()
X, y = smote.fit_resample(X, y)

print(Counter(y))

Counter({1: 500, 0: 500})


In [26]:
accuracy_precision_recall(X, y)

accuracy: 0.71
precision: 0.7196652719665272
recall: 0.688
f1_score: 0.7034764826175869


## Sobremuestreo ADASYN

Sobremuestreo Sintético Adaptativo (`ADASYN`). Es una evolución directa de SMOTE que utiliza un enfoque **adaptativo** priorizando las zonas del espacio de características donde el modelo de clasificación tiene mayor dificultad.

**Mecánica de Ponderación y Densidad:**

Para cada observación $x_i$ de la clase minoritaria, se calcula la proporción de vecinos de la clase mayoritaria dentro de su vecindad de $k$ vecinos:

$$r_i = \frac{\Delta_i}{k}$$

Donde $\Delta_i$ es el número de vecinos de la clase mayoritaria. Un $r_i$ cercano a $1$ indica una **zona de alta dificultad**. Se normalizan estos valores para obtener una distribución de probabilidad y el número de puntos sintéticos a generar para cada $x_i$ es proporcional a su nivel de dificultad.

---

**Resultado Final:** ADASYN **no siempre arroja un balance simétrico idéntico exacto** ($50\% - 50\%$) porque redondea según las ponderaciones calculadas:

$$\text{Counter}(\{0: 1000, 1: 996\})$$

> **Consideración Metodológica Crítica:** La gran ventaja de ADASYN es que reduce proactivamente el sesgo del desbalance obligando al clasificador a enfocarse en los ejemplos más complejos. Sin embargo, es **altamente vulnerable al ruido**: los *outliers* minoritarios inmersos en territorio de la clase mayoritaria serán interpretados como zonas de alta dificultad, creando una enorme cantidad de datos sintéticos alrededor de un error.

In [27]:
X = df[df.columns.drop("Outcome")]
y = df["Outcome"]

adasyn = im.over_sampling.ADASYN()
X, y = adasyn.fit_resample(X, y)

print(Counter(y))

Counter({0: 500, 1: 474})


In [28]:
accuracy_precision_recall(X, y)

accuracy: 0.6919917864476386
precision: 0.6790123456790124
recall: 0.6962025316455697
f1_score: 0.6875
